# Figure 7 — SNJ with subsampling

Three SNJ-paper metrics versus the subsampling fraction `p`, then `p*` vs `n`:

1. $\|R - \hat R\|_2$ (Thm 4.2)
2. $\sigma_2$ separation: distributions of $\Lambda(i,j)$ for adjacent vs. non-adjacent leaf pairs at step 0 of SNJ (Fig. 3 analog)
3. Robinson-Foulds distance from SNJ on $\hat R$

Baseline: uniform sampling with **no** matrix completion.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json
from datetime import datetime
from pathlib import Path

_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# Drop any stale cached copies of our project modules so this cell picks up
# edits made since the kernel started. autoreload handles subsequent edits.
for _mod in [m for m in list(sys.modules) if m.startswith(('scripts.', 'src.'))]:
    del sys.modules[_mod]

from src.config.presets import custom_config
from src.runners.snj_sweep import snj_sweep_for_params
from src.utils.snj_io import load_snj_results
from scripts.plot_snj_three_panel import _plot_three_panel
from scripts.plot_snj_p_star_vs_n import (
    _load_sweep, _compute_p_stars, _plot,
    _plot_overlay, _plot_sigma2_overlay,
)

## Choose: load or launch
Set `RUN_DIR` to an existing sweep directory to **load** prior results. Leave it `None` to **launch** a fresh sweep with the grid below.

In [ ]:
# Default: load an existing sweep. Set RUN_DIR=None to launch a fresh sweep
# using the TAXA/SEQLEN/REPS/P_VALS grid below.
RUN_DIR = Path(_root) / 'results' / 'runs' / '20260513-001823-snj_notebook'

TAXA   = [128, 256, 512, 1024, 2048, 4096]
SEQLEN = 2000
REPS   = 5
P_VALS = [1.0,  0.90, 0.70, 0.50, 0.30, 0.10, 0.005, 0.001, 0.0005, 0.0001]

if RUN_DIR is None:
    RUN_DIR = Path(_root) / 'results' / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S') + '-snj_notebook')
    for n in TAXA:
        cfg = custom_config(num_taxa=n, sequence_length=SEQLEN, mutation_rate=0.1,
                            tree_model='balanced_binary', seq_model='JC69',
                            p_values=P_VALS, bootstrap_reps=REPS,
                            sampling_method='uniform', matrix_kind='similarity')
        snj_sweep_for_params(cfg, n, SEQLEN, str(RUN_DIR / f'n{n}_L{SEQLEN}'))
else:
    RUN_DIR = Path(RUN_DIR)
print('sweep dir:', RUN_DIR)

## Three-panel plot, per n

In [ ]:
from IPython.display import Image, display
for sub in sorted(RUN_DIR.iterdir()):
    if not sub.is_dir():
        continue
    if not ((sub / 'snj_meta.json').exists() or (sub / 'snj_results.json').exists()):
        continue
    r = load_snj_results(sub)
    out = sub / 'snj_three_panel.png'
    _plot_three_panel(r, out)
    display(Image(filename=str(out)))

## p\* vs n

In [ ]:
rows = _compute_p_stars(_load_sweep(RUN_DIR))
_plot(rows, RUN_DIR / 'snj_p_star_vs_n.png')
_plot_overlay(RUN_DIR, RUN_DIR / 'snj_overlay_specnorm_rf.png')
_plot_sigma2_overlay(RUN_DIR, RUN_DIR / 'snj_overlay_sigma2.png')

for name in ('snj_overlay_specnorm_rf.png', 'snj_overlay_sigma2.png', 'snj_p_star_vs_n.png'):
    display(Image(filename=str(RUN_DIR / name)))
rows